**Tutorial series (1 of 4):**  [1 · Metagenomics](metagenomics.ipynb) · [2 · Build experimental DB](build_experimental_db.ipynb) · [3 · Parameter tuning](parameter_tuning.ipynb) · [4 · Modeling](modeling.ipynb)

# From an SRA table to a publication figure

This tutorial takes the Ding *et al.* 16S co-digestion samples **all the way from
an SRA accession table to the final NMDS + feature-enrichment figure**, using the
ADToolbox Python API one step at a time. The last section gives the equivalent
**one-command CLI** for the pipeline.

**Pipeline**

```
SRA table ──▶ download reads ──▶ trim + DADA2 (ASVs) ──▶ map ASVs to GTDB genomes
          ──▶ marker / COD allocation ──▶ per-sample cod_profile.csv + marker table
          ──▶ NMDS + PERMANOVA + enrichment  ──▶ ordination_panel.svg
```

The three feedstock groups are **FW+AS**, **FW+TWAS**, **FW+TWAS+AS** (food waste
co-digested with activated sludge, thickened WAS, or both), each with two day-0
replicates.


## 1. Setup

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import SVG, display

from adtoolbox import core, configs, markers

# --- paths (edit these for your machine) ---
REPO = Path.cwd().parent if Path.cwd().name == "Examples" else Path.cwd()
STUDY_TABLE  = REPO / "Examples" / "Studies" / "16s_sra_day0.csv"   # the SRA table
DATABASE_DIR = REPO / "database"                                    # ADToolbox databases (GTDB, ...)
OUTPUT_DIR   = REPO / "tutorial_output" / "ding_day0"               # per-sample results land here
FIG_DIR      = REPO / "tutorial_output" / "figs"
EXEC_PROFILE = REPO / "reference_data" / "metagenomics_pipeline.toml"

FIG_DIR.mkdir(parents=True, exist_ok=True)
print("study table:", STUDY_TABLE)
print("output dir :", OUTPUT_DIR)

## 2. The SRA sample table

The pipeline's only required input is a small CSV: one row per sample, with an
`accession` (SRA run) and a `sample_name`. Extra columns like `condition` are
carried along for our own grouping and ignored by the pipeline.

In [ ]:
manifest = pd.read_csv(STUDY_TABLE)
manifest

## 3. Run the pipeline: SRA → COD (the ADToolbox API)

`core.Metagenomics(...).batch_sample_to_cod(...)` runs every stage per sample:

| stage | what happens |
| --- | --- |
| `download`   | fetch each run from SRA (`prefetch` + `fasterq-dump`) |
| `preprocess` | quality-trim (fastp), remove 16S primers (cutadapt), infer ASVs (DADA2) |
| `allocate`   | map ASVs to GTDB representative genomes, annotate marker genes, and turn them into the e-ADM **COD-group** allocation |

> ⚠️ **This is a heavy step** — it downloads several GB and runs DADA2 + GTDB
> alignment + genome annotation (tens of minutes to hours), and needs the GTDB
> amplicon-to-genome database plus the external tools (`sra-tools`, `fastp`,
> `cutadapt`, DADA2/R, `vsearch`, `mmseqs2`, `ncbi-datasets-cli`). Set
> `RUN_PIPELINE = True` to actually run it; otherwise the notebook uses per-sample
> results already in `OUTPUT_DIR`.

In [ ]:
RUN_PIPELINE = False   # flip to True to actually download + process

config = configs.Metagenomics(metagenomics_dir=str(OUTPUT_DIR), database_dir=str(DATABASE_DIR))
mg = core.Metagenomics(config)

if RUN_PIPELINE:
    result = mg.batch_sample_to_cod(
        manifest=str(STUDY_TABLE),
        input_type="sra",            # the table holds SRA accessions
        assay="amplicon",            # 16S amplicon route
        output_dir=str(OUTPUT_DIR),
        stage="all",                 # download -> preprocess -> allocate
        amplicon_to_genome_db=str(DATABASE_DIR / "Amplicon2GenomeDBs"),
        genomes_dir=str(OUTPUT_DIR / "genomes"),
        execution_profile=str(EXEC_PROFILE),
        execute=True,
    )
    print("batch summary:", result["summary"])
else:
    print("RUN_PIPELINE is False — using existing per-sample outputs in", OUTPUT_DIR)

## 4. What the pipeline produced

Each sample gets its own folder. The two files we need are:

- `cod_profile.csv` — the model-ready **COD-group** composition (15 e-ADM groups)
- `sample_marker_abundances.csv` — the underlying **marker-gene** abundances (up to 96)

Both are *tall* tables (`sample, feature, value`).

In [ ]:
sample_dirs = sorted(p.name for p in OUTPUT_DIR.glob("*") if (p / "cod_profile.csv").exists())
print(f"{len(sample_dirs)} samples with COD profiles:")
print(" ", ", ".join(sample_dirs))

example = next(d for d in sample_dirs if d != "Blank")
print(f"\n--- {example}/cod_profile.csv ---")
display(pd.read_csv(OUTPUT_DIR / example / "cod_profile.csv").head())
print(f"--- {example}/sample_marker_abundances.csv ---")
display(pd.read_csv(OUTPUT_DIR / example / "sample_marker_abundances.csv")
        [["sample", "marker_id", "weighted_abundance"]].head())

## 5. Analysis utilities

The helpers below turn the per-sample tables into matrices and run the ordination,
PERMANOVA, enrichment, and figure. They are plain NumPy / SciPy / scikit-learn /
Matplotlib — you can read or tweak any of them here. On a first pass you can run
this cell and move on; the science is called out step by step in sections 6–9.

In [ ]:
import glob
import os
import re

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgba

LEVELS = {
    "cod": ("cod_profile.csv", "group", "value"),
    "marker": ("sample_marker_abundances.csv", "marker_id", "weighted_abundance"),
}
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00",
           "#56B4E9", "#F0E442", "#999999", "#000000"]


# ---------------------------------------------------------------------- styling

def set_pub_style() -> None:
    plt.rcParams.update({
        "figure.dpi": 150, "savefig.dpi": 400, "savefig.bbox": "tight",
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
        "axes.titleweight": "semibold", "axes.linewidth": 1.1,
        "axes.edgecolor": "#2b2b2b", "xtick.color": "#2b2b2b", "ytick.color": "#2b2b2b",
        "xtick.direction": "out", "ytick.direction": "out",
        "legend.frameon": False, "svg.fonttype": "none",
    })


def style_axes(ax, equal=True) -> None:
    ax.spines[["top", "right"]].set_visible(False)
    ax.axhline(0, color="#9e9e9e", lw=0.7, ls=(0, (4, 4)), alpha=0.7, zorder=0)
    ax.axvline(0, color="#9e9e9e", lw=0.7, ls=(0, (4, 4)), alpha=0.7, zorder=0)
    if equal:
        ax.set_aspect("equal", adjustable="datalim")
    ax.margins(0.16)


def panel_title(ax, letter, title, stats):
    ax.set_title(f"{title}", pad=24, loc="center")
    ax.annotate(letter, xy=(0.0, 1.0), xycoords="axes fraction", xytext=(-38, 14),
                textcoords="offset points", fontsize=14, fontweight="bold", va="bottom")
    ax.text(0.5, 1.012, stats, transform=ax.transAxes, ha="center", va="bottom",
            fontsize=8.5, color="#555555")


# --------------------------------------------------------------------------- IO

def load_level(cod_dir, level, drop):
    filename, feature_col, value_col = LEVELS[level]
    paths = sorted(glob.glob(os.path.join(cod_dir, "*", filename)))
    if not paths:
        print(f"[{level}] no <sample>/{filename} under {cod_dir!r} - skipping")
        return None
    tall = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    wide = tall.pivot_table(index="sample", columns=feature_col, values=value_col,
                            aggfunc="sum", fill_value=0.0)
    wide = wide.drop(index=[s for s in drop if s in wide.index], errors="ignore")
    wide = wide.loc[wide.sum(axis=1) > 0]
    wide = wide.loc[:, wide.sum(axis=0) > 0]
    return wide


def group_of(sample, manifest):
    if manifest and sample in manifest:
        return manifest[sample]
    return re.sub(r"_rep[_-]?\d+.*$", "", sample).replace("_", "+")


def read_manifest(path):
    m = pd.read_csv(path)
    nc = next(c for c in m.columns if c.lower() in ("sample_name", "sample", "name"))
    gc = next(c for c in m.columns if c.lower() in ("condition", "group"))
    return dict(zip(m[nc].astype(str), m[gc].astype(str)))


# --------------------------------------------------------------------- analyses

def run_nmds(wide):
    dist = squareform(pdist(wide.to_numpy(float), metric="braycurtis"))
    kw = dict(n_components=2, metric=False, dissimilarity="precomputed",
              random_state=0, n_init=10, max_iter=500)
    try:
        nmds = MDS(normalized_stress="auto", **kw)
    except TypeError:
        nmds = MDS(**kw)
    return nmds.fit_transform(dist), float(nmds.stress_), dist


def permanova(dist, groups, permutations=999, seed=0):
    """Distance-based PERMANOVA (Anderson 2001). Returns pseudo-F, R^2, p."""
    g = np.asarray(groups)
    n = len(g)
    d2 = dist ** 2
    levels = np.unique(g)
    a = len(levels)
    sst = d2[np.triu_indices(n, 1)].sum() / n

    def ssw(labels):
        s = 0.0
        for lev in levels:
            idx = np.where(labels == lev)[0]
            k = len(idx)
            if k > 1:
                sub = d2[np.ix_(idx, idx)]
                s += sub[np.triu_indices(k, 1)].sum() / k
        return s

    w = ssw(g)
    fstat = ((sst - w) / (a - 1)) / (w / (n - a)) if (n - a) > 0 and w > 0 else np.nan
    r2 = (sst - w) / sst if sst > 0 else np.nan
    rng = np.random.default_rng(seed)
    ge = 1
    for _ in range(permutations):
        wp = ssw(rng.permutation(g))
        fp = ((sst - wp) / (a - 1)) / (wp / (n - a)) if wp > 0 else np.inf
        if fp >= fstat:
            ge += 1
    return fstat, r2, ge / (permutations + 1)


def run_rda(wide, groups, permutations=999, seed=0):
    """Hellinger-RDA constrained by group + permutation F test.

    Returns site scores, feature loadings, per-axis variance fractions, and a
    dict of stats {constrained, F, p}.
    """
    y = wide.to_numpy(float)
    y = np.sqrt(y / y.sum(axis=1, keepdims=True))
    yc = y - y.mean(axis=0)
    total = float((yc ** 2).sum())
    levels = np.unique(groups)
    a = len(levels)
    n = len(groups)

    def fit_ss(labels):
        g = pd.get_dummies(list(labels)).to_numpy(float)
        gc = g - g.mean(axis=0)
        fitted = gc @ (np.linalg.pinv(gc) @ yc)
        return fitted, float((fitted ** 2).sum())

    fitted, ssc = fit_ss(groups)
    ssr = total - ssc
    dfc, dfr = a - 1, n - a
    fstat = (ssc / dfc) / (ssr / dfr) if dfr > 0 and ssr > 0 else np.nan
    rng = np.random.default_rng(seed)
    ge = 1
    for _ in range(permutations):
        _, sscp = fit_ss(rng.permutation(np.asarray(groups)))
        fp = (sscp / dfc) / ((total - sscp) / dfr) if (total - sscp) > 0 else np.inf
        if fp >= fstat:
            ge += 1
    u, s, vt = np.linalg.svd(fitted, full_matrices=False)
    k = min(2, s.shape[0])
    site, load = u[:, :k] * s[:k], vt[:k, :].T
    if k == 1:
        site = np.column_stack([site, np.zeros(len(site))])
        load = np.column_stack([load, np.zeros(len(load))])
    stats = {"constrained": ssc / total if total > 0 else 0.0,
             "F": fstat, "p": ge / (permutations + 1)}
    return site, load, (s ** 2) / total, stats


# ------------------------------------------------------------------------ panels

def _nudge_coincident(coords, frac=0.045):
    """Spread near-coincident points slightly so overlapping replicates are both
    visible. Only points closer than frac*span are moved (radially around their
    shared centre); well-separated points are untouched."""
    out = np.asarray(coords, float).copy()
    n = len(out)
    span = max(np.ptp(out[:, 0]), np.ptp(out[:, 1])) or 1.0
    thr = frac * span
    seen = [False] * n
    for i in range(n):
        if seen[i]:
            continue
        cluster = [j for j in range(n) if np.hypot(*(out[j] - out[i])) < thr]
        for j in cluster:
            seen[j] = True
        if len(cluster) > 1:
            centre = out[cluster].mean(axis=0)
            r = thr * 0.7
            for k, j in enumerate(cluster):
                ang = 2 * np.pi * k / len(cluster)
                out[j] = centre + r * np.array([np.cos(ang), np.sin(ang)])
    return out


def draw_points(ax, coords, groups, colours):
    # Translucent fill + solid thin edge, and a small nudge so coincident
    # replicates show as two dots rather than one hiding the other.
    coords = _nudge_coincident(coords)
    for g in sorted(set(groups)):
        idx = [i for i, gg in enumerate(groups) if gg == g]
        ax.scatter(coords[idx, 0], coords[idx, 1], s=110, marker="o",
                   facecolor=to_rgba(colours[g], 0.55), edgecolor=colours[g],
                   linewidth=1.2, zorder=4)


def fmt_p(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


def nmds_panel(ax, coords, groups, stress, perm, level, colours, letter):
    draw_points(ax, coords, groups, colours)
    ax.set_xlabel("NMDS1"); ax.set_ylabel("NMDS2")
    f, r2, p = perm
    panel_title(ax, letter, f"NMDS · {level.upper()} · Bray–Curtis",
                f"stress = {stress:.3f}   |   PERMANOVA pseudo-F = {f:.2f}, "
                f"R² = {r2:.2f}, {fmt_p(p)}")
    style_axes(ax, equal=True)


def rda_panel(ax, site, load, features, groups, axis_frac, stats, level, colours, letter, top_n):
    draw_points(ax, site, groups, colours)
    mag = np.hypot(load[:, 0], load[:, 1])
    top = np.argsort(mag)[::-1][:top_n]
    sx, sy = np.abs(site[:, 0]).max() or 1.0, np.abs(site[:, 1]).max() or 1.0
    lx, ly = np.abs(load[top, 0]).max() or 1.0, np.abs(load[top, 1]).max() or 1.0
    scale = 0.9 * min(sx / lx, (sy / ly) if ly > 1e-9 else np.inf)
    for j in top:
        dx, dy = load[j, 0] * scale, load[j, 1] * scale
        ax.annotate("", xy=(dx, dy), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="#b22222", lw=1.2, alpha=0.8), zorder=6)
        ax.annotate(str(features[j]), (dx * 1.06, dy * 1.06), fontsize=8, color="#8b1a1a",
                    ha="left" if dx >= 0 else "right", va="bottom" if dy >= 0 else "top",
                    zorder=7, bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.65))
    ax.set_xlabel(f"RDA1 ({axis_frac[0] * 100:.0f}%)")
    ax.set_ylabel(f"RDA2 ({axis_frac[1] * 100:.0f}%)" if len(axis_frac) > 1 else "RDA2")
    panel_title(ax, letter, f"RDA · {level.upper()} · constrained by group",
                f"constrained = {stats['constrained'] * 100:.0f}%   |   "
                f"F = {stats['F']:.2f}, {fmt_p(stats['p'])}")
    style_axes(ax, equal=False)


def shared_legend(fig, colours):
    handles = [Line2D([0], [0], marker="o", linestyle="", markerfacecolor=c,
                      markeredgecolor="white", markersize=9, label=g)
               for g, c in colours.items()]
    fig.legend(handles=handles, title="Group", loc="lower center",
               ncol=min(len(handles), 4), bbox_to_anchor=(0.5, -0.02),
               title_fontproperties={"weight": "semibold"})


# ---------------------------------------------------------------- enrichment

def compute_enrichment(wide, groups):
    """Per-group feature means and their across-group z-scores (enrichment)."""
    g = pd.Series(list(groups), index=wide.index)
    gmean = wide.groupby(g).mean().sort_index()                 # groups x features
    z = (gmean - gmean.mean(axis=0)) / (gmean.std(axis=0) + 1e-12)
    return gmean, z


def export_enrichment(gmean, z, path):
    out = gmean.T.copy()
    out.columns = [f"mean_{c}" for c in out.columns]
    zt = z.T.copy()
    zt.columns = [f"z_{c}" for c in zt.columns]
    out = out.join(zt)
    others = {g: gmean.drop(index=g).mean(axis=0) for g in gmean.index}
    enr = z.T.idxmax(axis=1)
    out.insert(0, "enriched_group", enr)
    out["log2FC_vs_others"] = [
        np.log2((gmean.loc[enr[f], f] + 1e-9) / (others[enr[f]][f] + 1e-9))
        for f in out.index
    ]
    out.index.name = "feature"
    out.sort_values(["enriched_group", "log2FC_vs_others"], ascending=[True, False]).to_csv(path)


def enrichment_panel(ax, gmean, z, level, letter, colours, top_n):
    zt = z.T                                                    # features x groups
    if top_n and zt.shape[0] > top_n:                          # keep most variable
        keep = gmean.std(axis=0).sort_values(ascending=False).index[:top_n]
        zt = zt.loc[keep]
    enr = np.asarray(zt.values).argmax(axis=1)
    order = sorted(range(zt.shape[0]), key=lambda i: (enr[i], -zt.values[i, enr[i]]))
    zt = zt.iloc[order]
    groups_order = list(zt.columns)
    vmax = float(np.nanmax(np.abs(zt.values))) or 1.0
    im = ax.imshow(zt.values, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(groups_order)))
    ax.set_xticklabels(groups_order, rotation=25, ha="right", fontsize=9)
    for tick, g in zip(ax.get_xticklabels(), groups_order):
        tick.set_color(colours.get(g, "#333333")); tick.set_fontweight("semibold")
    ax.set_yticks(range(zt.shape[0]))
    ax.set_yticklabels(list(zt.index), fontsize=7.5)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(f"Enrichment · {level.upper()}", pad=20, loc="center")
    ax.annotate(letter, xy=(0.0, 1.0), xycoords="axes fraction", xytext=(-46, 12),
                textcoords="offset points", fontsize=14, fontweight="bold", va="bottom")
    ax.text(0.5, 1.01, "row z-score of group-mean abundance", transform=ax.transAxes,
            ha="center", va="bottom", fontsize=8.5, color="#555555")
    return im


# ------------------------------------------------------------------------- driver

def build_panel(cod_dir, *, manifest=None, only_manifest=False, drop=("Blank",),
                level="both", second_row="enrichment", permutations=999,
                top_features=10, top_markers=25, outdir=".", fmt="svg", verbose=True):
    """Build the ordination panel from a pipeline output directory.

    This is the single code path used by both the CLI and the tutorial notebook.
    Returns the path to the written figure. `manifest` may be a CSV path or a
    {sample_name: group} mapping.
    """
    set_pub_style()
    os.makedirs(outdir, exist_ok=True)
    if isinstance(manifest, (str, os.PathLike)):
        manifest = read_manifest(manifest)
    levels = ["cod", "marker"] if level == "both" else [level]
    second = second_row
    log = print if verbose else (lambda *a, **k: None)

    results, all_groups = {}, set()
    for lv in levels:
        wide = load_level(cod_dir, lv, list(drop))
        if wide is None:
            continue
        if only_manifest:
            if not manifest:
                raise ValueError("only_manifest=True requires a manifest")
            wide = wide.loc[[s for s in wide.index if s in manifest]]
        samples = list(wide.index)
        groups = [group_of(s, manifest) for s in samples]
        if len(samples) < 3 or len(set(groups)) < 2:
            log(f"[{lv}] need >=3 samples and >=2 groups; have "
                f"{len(samples)} samples / {len(set(groups))} groups - skipping")
            continue
        all_groups.update(groups)
        coords, stress, dist = run_nmds(wide)
        perm = permanova(dist, groups, permutations)
        entry = {"samples": samples, "groups": groups, "coords": coords,
                 "stress": stress, "perm": perm}
        log(f"[{lv}] n={len(samples)} | NMDS stress={stress:.3f} | "
            f"PERMANOVA F={perm[0]:.2f} R2={perm[1]:.2f} p={perm[2]:.3f}")
        if second == "rda":
            site, load, axis_frac, rstats = run_rda(wide, groups, permutations)
            entry.update(site=site, load=load, features=list(wide.columns),
                         axis_frac=axis_frac, rstats=rstats)
            log(f"[{lv}] RDA constrained={rstats['constrained']*100:.0f}% "
                f"F={rstats['F']:.2f} p={rstats['p']:.3f}")
        elif second == "enrichment":
            gmean, z = compute_enrichment(wide, groups)
            entry.update(gmean=gmean, z=z)
            csv = os.path.join(outdir, f"enrichment_{lv}.csv")
            export_enrichment(gmean, z, csv)
            enr = z.T.idxmax(axis=1)
            for g in sorted(set(groups)):
                top = z.T[enr == g].loc[:, g].sort_values(ascending=False).head(5).index.tolist()
                log(f"[{lv}] enriched in {g}: {', '.join(map(str, top))}")
        results[lv] = entry

    if not results:
        raise SystemExit("No level had enough samples to ordinate. "
                         "Has the pipeline finished producing per-sample cod_profile.csv?")

    colours = {g: PALETTE[i % len(PALETTE)] for i, g in enumerate(sorted(all_groups))}
    plevels = [lv for lv in levels if lv in results]
    have_second = second != "none"
    ncols = len(plevels)
    nrows = 2 if have_second else 1
    want_cbar = have_second and second == "enrichment"
    # Dedicated thin last column for the colorbar so no panel loses width; the
    # panel columns then share identical x-extents across rows (aligned centres).
    fig = plt.figure(figsize=(6.7 * ncols + (0.8 if want_cbar else 0.2), 5.9 * nrows))
    width_ratios = [1] * ncols + ([0.05] if want_cbar else [])
    gs = fig.add_gridspec(nrows, len(width_ratios), width_ratios=width_ratios,
                          wspace=0.5, hspace=0.5)
    axes = np.empty((nrows, ncols), dtype=object)
    for r_i in range(nrows):
        for c_i in range(ncols):
            axes[r_i, c_i] = fig.add_subplot(gs[r_i, c_i])

    letters = iter("abcdefgh")
    for j, lv in enumerate(plevels):
        r = results[lv]
        nmds_panel(axes[0, j], r["coords"], r["groups"], r["stress"], r["perm"],
                   lv, colours, next(letters))
    heat_ims = []
    if have_second:
        for j, lv in enumerate(plevels):
            r = results[lv]
            if second == "rda":
                rda_panel(axes[1, j], r["site"], r["load"], r["features"], r["groups"],
                          r["axis_frac"], r["rstats"], lv, colours, next(letters), top_features)
            else:
                heat_ims.append(enrichment_panel(axes[1, j], r["gmean"], r["z"], lv,
                                                 next(letters), colours, top_markers))
    if want_cbar and heat_ims:
        cbar = fig.colorbar(heat_ims[-1], cax=fig.add_subplot(gs[nrows - 1, ncols]))
        cbar.set_label("row z-score  (red = enriched)", fontsize=9)
    shared_legend(fig, colours)
    out = os.path.join(outdir, f"ordination_panel.{fmt}")
    fig.savefig(out); plt.close(fig)
    log(f"wrote {out}")
    return out

## 6. Build the sample × feature matrices

`load_level` gathers every `<sample>/<file>` into a samples × feature matrix,
drops the `Blank` control, and drops features never observed. We then restrict to
the samples in our table and label each sample by its group.

In [ ]:
groups_map = read_manifest(str(STUDY_TABLE))       # {sample_name: condition}

cod = load_level(str(OUTPUT_DIR), "cod", drop=["Blank"])
mk  = load_level(str(OUTPUT_DIR), "marker", drop=["Blank"])
cod = cod.loc[[s for s in cod.index if s in groups_map]]
mk  = mk.loc[[s for s in mk.index if s in groups_map]]

groups = [group_of(s, groups_map) for s in cod.index]
print("samples:", list(cod.index))
print("groups :", groups)
print(f"COD matrix: {cod.shape[0]} samples x {cod.shape[1]} groups")
print(f"marker matrix: {mk.shape[0]} samples x {mk.shape[1]} genes")
cod.round(3)

## 7. Ordination: Bray–Curtis + NMDS, tested with PERMANOVA

`run_nmds` computes a Bray–Curtis distance matrix and runs non-metric MDS.
`permanova` tests whether the groups differ (distance-based, permutation p).
With only 2 replicates per group the p-value is at its permutation floor
(~0.02) — so report the **R² effect size**, not just p.

In [ ]:
for name, wide in [("COD", cod), ("marker", mk)]:
    coords, stress, dist = run_nmds(wide)
    F, R2, p = permanova(dist, groups, permutations=999)
    print(f"{name:6}  stress={stress:.3f}   PERMANOVA pseudo-F={F:.1f}  R2={R2:.2f}  p={p:.3f}")

## 8. Enrichment: which features characterise each group

For each feature we take the per-group mean and standardise it across groups
(row z-score). A feature's *enriched group* is where that z-score is highest.

In [ ]:
gmean, z = compute_enrichment(cod, groups)
enriched = z.T.idxmax(axis=1)
for g in sorted(set(groups)):
    top = z.T[enriched == g].loc[:, g].sort_values(ascending=False).head(6).index.tolist()
    print(f"COD enriched in {g:12}: {', '.join(top)}")

## 9. Assemble the publication figure

`build_panel` wraps sections 6–8 for both feature levels and lays them out as the
aligned panel — NMDS on top (a, b), enrichment heatmaps below (c, d), a shared
colour legend, and a colorbar in its own column. It writes `ordination_panel.svg`
and the two `enrichment_*.csv` tables.

In [ ]:
out = build_panel(
    str(OUTPUT_DIR),
    manifest=str(STUDY_TABLE),
    only_manifest=True,        # keep just the samples in the table
    drop=["Blank"],
    level="both",              # COD and marker
    second_row="enrichment",   # bottom row = enrichment heatmaps (use "rda" for RDA)
    outdir=str(FIG_DIR),
    fmt="svg",
)
display(SVG(filename=out))

## 10. The pipeline in one command

Sections 2–4 (SRA → per-sample COD) are a single CLI call:

```bash
adtoolbox metagenomics process \
  --input Examples/Studies/16s_sra_day0.csv \
  --input-type sra \
  --assay amplicon \
  --database-dir ./database \
  --amplicon-to-genome-db ./database/Amplicon2GenomeDBs \
  --output-dir ./tutorial_output/ding_day0 \
  --stage all \
  --execution-profile reference_data/metagenomics_pipeline.toml \
  --execute
```

The ordination + enrichment figure (sections 6–9) is produced by the analysis
cells above; point `OUTPUT_DIR` at the directory this command wrote.


## Notes & caveats

- **Statistics:** with 6 samples (2 reps × 3 groups) the PERMANOVA p-value cannot
  drop below ~0.02 (limited permutations). Emphasise R² / the enrichment pattern.
  Including the day-24 samples (n=12) gives real power — rerun sections 6–9 on the
  full output directory (drop `only_manifest`, or use a 12-sample table).
- **Enrichment is descriptive**, not a per-feature significance test — with 2
  replicates a Kruskal–Wallis / DESeq test would be invalid. The exported CSV's
  `log2FC_vs_others` gives the effect size.
- **Reproducibility:** the pipeline stamps the marker-catalog version and caches
  by input signature, so re-running reuses completed work instead of redoing it.
